# IPL Auction Inefficiency: Identifying Mispriced Players in the IPL Auction Market

This notebook treats the IPL auction as a noisy, imperfect talent market. Auction price is the observed market price, model-estimated fair price is intrinsic value, and the difference between the two is our mispricing signal.

**Core thesis:**
- Positive mispricing means the player was undervalued.
- Negative mispricing means the player was overvalued.
- We focus on modern IPL auctions and primarily map auction year `Y` to player performance in season `Y-1`.

## 1. Project setup and imports

The code below resolves the project root dynamically, imports the reusable modules from `src/`, and ensures output folders exist.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'src').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'src').exists() and (parent / 'data').exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import discover_raw_files, identify_concept_files, load_concept_tables, profile_raw_files
from src.cleaning import (
    combine_auction_sources,
    harmonize_auction_to_performance_names,
    standardize_deliveries_data,
    standardize_matches_data,
)
from src.features import build_pre_auction_dataset, build_player_season_features, build_realized_season_review
from src.modeling import add_mispricing_columns, model_comparison_table, prepare_model_dataset, select_primary_model, train_models
from src.utils import ensure_directories
from src.visuals import (
    plot_franchise_efficiency,
    plot_price_vs_predicted,
    plot_quadrant,
    plot_role_boxplot,
    plot_top_mispricing,
    set_matplotlib_theme,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_CHARTS = PROJECT_ROOT / 'outputs' / 'charts'

ensure_directories([DATA_PROCESSED, OUTPUT_TABLES, OUTPUT_CHARTS])
set_matplotlib_theme()

PROJECT_ROOT

## 2. File discovery and raw data preview

The project is intentionally defensive about raw inputs. Instead of assuming exact filenames, it scans `data/raw/` recursively, attempts to read likely CSV/XLSX files, and classifies them into conceptual buckets.

In [ ]:
files_df = discover_raw_files(DATA_RAW)
print(f'Raw files discovered: {len(files_df)}')
files_df

In [ ]:
profiles_df = profile_raw_files(files_df) if not files_df.empty else pd.DataFrame()
profiles_df

## 3. Cleaning and schema standardization

We normalize schemas to lowercase snake case, map common aliases, and then combine conceptually similar files. If both auction datasets are present, the expanded multi-year auction file is used as the base while duplicates are removed on player, auction year, and team.

In [ ]:
concept_files = identify_concept_files(files_df) if not files_df.empty else {'auction': [], 'matches': [], 'deliveries': []}
concept_files

In [ ]:
loaded = load_concept_tables(files_df) if not files_df.empty else {'auction': [], 'matches': [], 'deliveries': []}

auction_df = combine_auction_sources(loaded.get('auction', [])) if loaded.get('auction') else pd.DataFrame()
matches_df = pd.concat([standardize_matches_data(df) for df in loaded.get('matches', [])], ignore_index=True, sort=False) if loaded.get('matches') else pd.DataFrame()
deliveries_df = pd.concat([standardize_deliveries_data(df) for df in loaded.get('deliveries', [])], ignore_index=True, sort=False) if loaded.get('deliveries') else pd.DataFrame()

print('Auction rows:', len(auction_df))
print('Matches rows:', len(matches_df))
print('Deliveries rows:', len(deliveries_df))

auction_df.head()

## 4. Player name harmonization

IPL data is messy enough that name cleaning is a project by itself. The pipeline uses canonical lowercase names, punctuation stripping, a manual map for common IPL variants, and conservative fuzzy matching for unresolved rows. Low-confidence suggestions are kept in a review table rather than forced into the analysis.

In [ ]:
player_features_seed = build_player_season_features(deliveries_df, matches_df) if not deliveries_df.empty and not matches_df.empty else pd.DataFrame()
harmonized_auction_df, name_review_df = harmonize_auction_to_performance_names(auction_df, player_features_seed) if not auction_df.empty else (pd.DataFrame(), pd.DataFrame())

print('Performance player-season rows:', len(player_features_seed))
print('Uncertain name review rows:', len(name_review_df))
name_review_df.head(20)

## 5. Match-season linkage

Every delivery needs to be tied back to an IPL season using the matches table. This is essential because the auction-year mapping depends on season-specific performance rather than pooled career stats.

In [ ]:
player_season_features = player_features_seed.copy()
player_season_features.head()

## 6. Batting and bowling feature engineering

The feature set includes batting output, strike rate, boundary rate, dot-ball pressure, phase splits, wickets, economy, bowling strike rate, and experience proxies. These are all computed from the season before the auction whenever the year mapping is clean.

In [ ]:
selected_columns = [
    'match_season', 'player_name_clean', 'runs_scored', 'batting_strike_rate', 'wickets',
    'economy_rate', 'experience_proxy', 'recent_runs_scored', 'recent_wickets'
]
available_columns = [col for col in selected_columns if col in player_season_features.columns]
player_season_features[available_columns].head(15) if not player_season_features.empty else player_season_features

## 7. Auction price cleaning and pre-auction merge

Price fields are standardized to INR and crore INR. The merge keys are auction year `Y`, prior performance season `Y-1`, and the harmonized player name. Ambiguous rows are dropped gracefully instead of forcing a bad join.

In [ ]:
merged_df, coverage_report = build_pre_auction_dataset(harmonized_auction_df, player_season_features) if not harmonized_auction_df.empty else (pd.DataFrame(), pd.DataFrame())
coverage_report

In [ ]:
merged_df.head()

## 8. Exploratory analysis

Before modeling, it helps to understand the auction market directly: how market size changes over time, how pricing varies by role, and which observable features appear most related to price.

In [ ]:
auction_market_by_year = pd.DataFrame()
price_distribution_by_role = pd.DataFrame()
feature_correlations = pd.DataFrame()

if not merged_df.empty:
    auction_market_by_year = merged_df.groupby('auction_year', dropna=False).agg(
        players_sold=('player_name', 'count'),
        total_spend_inr=('price_in_inr', 'sum'),
        median_price_inr=('price_in_inr', 'median'),
    ).reset_index()
    price_distribution_by_role = merged_df.groupby('role_bucket', dropna=False).agg(
        players=('player_name', 'count'),
        average_price_crore=('price_in_crore', 'mean'),
        median_price_crore=('price_in_crore', 'median'),
    ).reset_index().sort_values('average_price_crore', ascending=False)
    numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()
    if 'price_in_inr' in numeric_cols:
        corr = merged_df[numeric_cols].corr()['price_in_inr'].dropna().sort_values(ascending=False)
        feature_correlations = corr.reset_index()
        feature_correlations.columns = ['feature', 'correlation_with_price']

auction_market_by_year

In [ ]:
price_distribution_by_role

In [ ]:
feature_correlations.head(20)

## 9. Modeling

We compare three valuation approaches:

1. An explainable role-aware scorecard
2. Linear regression on log prices
3. Tree-based regression on log prices

The holdout split is by auction year rather than random row sampling to avoid temporal leakage.

In [ ]:
model_df = prepare_model_dataset(merged_df, min_year=2018) if not merged_df.empty else pd.DataFrame()
model_bundles = train_models(model_df) if not model_df.empty else {}
comparison_df = model_comparison_table(model_bundles) if model_bundles else pd.DataFrame()
comparison_df

In [ ]:
primary_model = select_primary_model(model_bundles) if model_bundles else None
primary_model.name if primary_model else 'No model trained'

## 10. Mispricing analysis

The selected model produces the final fair-price estimate. Mispricing is calculated as model-implied value minus actual auction price, so positive values indicate that the player looked underpriced relative to measurable prior performance.

In [ ]:
final_model_dataset = add_mispricing_columns(model_df, primary_model.predictions, primary_model.name) if primary_model else pd.DataFrame()

player_mispricing_table = final_model_dataset.sort_values('mispricing_in_inr', ascending=False) if not final_model_dataset.empty else pd.DataFrame()
top_undervalued_players = player_mispricing_table.head(15).copy() if not player_mispricing_table.empty else pd.DataFrame()
top_overvalued_players = player_mispricing_table.sort_values('mispricing_in_inr').head(15).copy() if not player_mispricing_table.empty else pd.DataFrame()
article_2025_dataset = final_model_dataset.loc[final_model_dataset['auction_year'].eq(2025)].copy() if not final_model_dataset.empty else pd.DataFrame()
if not article_2025_dataset.empty:
    article_2025_dataset = article_2025_dataset.loc[
        article_2025_dataset['actual_price_in_crore'].fillna(0).ge(1.0)
        & (
            article_2025_dataset['matches_played'].fillna(0).ge(5)
            | article_2025_dataset['prior_auction_count'].fillna(0).ge(1)
            | article_2025_dataset['captaincy_proxy'].fillna(0).eq(1)
        )
    ].copy()
article_2025_player_table = article_2025_dataset.sort_values('article_mispricing_in_inr', ascending=False) if not article_2025_dataset.empty else pd.DataFrame()
article_top_5_undervalued_2025 = article_2025_player_table.head(5).copy() if not article_2025_player_table.empty else pd.DataFrame()
article_top_5_overvalued_2025 = article_2025_player_table.sort_values('article_mispricing_in_inr').head(5).copy() if not article_2025_player_table.empty else pd.DataFrame()
article_overvalued_2025_min10 = pd.DataFrame()
article_undervalued_2025_min10 = pd.DataFrame()

player_mispricing_table.head(20)

In [ ]:
role_mispricing_summary = pd.DataFrame()
overseas_summary = pd.DataFrame()

if not final_model_dataset.empty:
    role_mispricing_summary = final_model_dataset.groupby('role_bucket', dropna=False).agg(
        players=('player_name', 'count'),
        average_mispricing_inr=('mispricing_in_inr', 'mean'),
        median_mispricing_inr=('mispricing_in_inr', 'median'),
    ).reset_index().sort_values('average_mispricing_inr', ascending=False)
    overseas_summary = final_model_dataset.groupby('is_overseas', dropna=False).agg(
        players=('player_name', 'count'),
        average_price_crore=('actual_price_in_crore', 'mean'),
        average_mispricing_crore=('mispricing_in_inr', lambda s: s.mean() / 10_000_000.0),
    ).reset_index()
    overseas_summary['player_group'] = overseas_summary['is_overseas'].map({1: 'Overseas', 0: 'Indian'})

role_mispricing_summary

## 11. Franchise analysis

A player-level edge becomes a strategy edge only if franchises consistently capture positive mispricing. We summarize this as average fair value captured and a simple efficiency lens based on value bought per rupee spent.

In [ ]:
franchise_value_summary = pd.DataFrame()

if not final_model_dataset.empty:
    franchise_value_summary = final_model_dataset.groupby('team', dropna=False).agg(
        players_bought=('player_name', 'count'),
        total_spend_inr=('price_in_inr', 'sum'),
        avg_positive_mispricing_captured=('mispricing_in_inr', 'mean'),
        median_value_per_rupee=('mispricing_pct', 'median'),
    ).reset_index().sort_values('avg_positive_mispricing_captured', ascending=False)

franchise_value_summary

## 12. Case studies

A few player-level narratives make the model easier to interpret. These are intentionally concise and are meant to translate the table output into a scouting and auction strategy lens.

In [ ]:
case_studies = []
if not final_model_dataset.empty:
    pant_priority = final_model_dataset.loc[final_model_dataset['player_name'].eq('Rishabh Pant') & final_model_dataset['auction_year'].eq(2025)]
    notable_players = final_model_dataset.reindex(final_model_dataset['mispricing_in_inr'].abs().sort_values(ascending=False).head(8).index)
    if not pant_priority.empty:
        notable_players = pd.concat([pant_priority, notable_players], ignore_index=False)
    notable_players = notable_players.drop_duplicates(subset=['player_name', 'auction_year']).head(3)
    for _, row in notable_players.iterrows():
        direction = 'undervalued' if row['mispricing_in_inr'] > 0 else 'overvalued'
        case_studies.append({
            'player_name': row['player_name'],
            'auction_year': row['auction_year'],
            'team': row.get('team', 'Unknown'),
            'narrative': (
                f"{row['player_name']} looked {direction} in {int(row['auction_year']) if pd.notna(row['auction_year']) else 'the auction year'}: "
                f"actual price was {row['actual_price_in_crore']:.2f} crore while the model estimated {row['predicted_price_in_crore']:.2f} crore."
            ),
        })
case_study_df = pd.DataFrame(case_studies)
case_study_df

## 13. 2025 Reality Check

A market model is most useful when we compare it against what happened next. This section focuses only on the top five undervalued and overvalued players from the 2025 auction sample, then checks how those players actually performed in the 2025 IPL season.

In [ ]:
top_5_undervalued_2025_realized_review = pd.DataFrame()
top_5_overvalued_2025_realized_review = pd.DataFrame()
article_min10_2025_dataset = pd.DataFrame()

if not article_2025_dataset.empty and not player_season_features.empty:
    top_5_undervalued_2025_realized_review, top_5_overvalued_2025_realized_review = build_realized_season_review(
        final_model_dataset=article_2025_dataset,
        player_features=player_season_features,
        matches_df=matches_df,
        auction_year=2025,
        realized_season=2025,
        top_n=5,
    )

    if not top_5_undervalued_2025_realized_review.empty:
        top_5_undervalued_2025_realized_review = top_5_undervalued_2025_realized_review.sort_values('article_mispricing_in_inr', ascending=False).head(5)
    if not top_5_overvalued_2025_realized_review.empty:
        top_5_overvalued_2025_realized_review = top_5_overvalued_2025_realized_review.sort_values('article_mispricing_in_inr', ascending=True).head(5)
    realized_pool = article_2025_dataset.copy()
    realized_pool['player_name_join'] = realized_pool.get('matched_player_name_clean', realized_pool.get('player_name_clean', realized_pool['player_name']))
    realized_2025 = player_season_features.loc[player_season_features['match_season'].eq(2025), ['player_name_clean', 'matches_played', 'runs_scored', 'batting_strike_rate', 'wickets', 'economy_rate', 'bowling_strike_rate']].copy()
    realized_2025 = realized_2025.rename(columns={'player_name_clean': 'player_name_join', 'matches_played': 'realized_matches_played', 'runs_scored': 'runs_scored_2025', 'batting_strike_rate': 'batting_strike_rate_2025', 'wickets': 'wickets_2025', 'economy_rate': 'economy_rate_2025', 'bowling_strike_rate': 'bowling_strike_rate_2025'})
    article_min10_2025_dataset = realized_pool.merge(realized_2025, on='player_name_join', how='left')
    article_min10_2025_dataset = article_min10_2025_dataset.loc[
        article_min10_2025_dataset['actual_price_in_crore'].fillna(0).ge(1.0)
        & article_min10_2025_dataset['realized_matches_played'].fillna(0).ge(10)
    ].copy()
    article_undervalued_2025_min10 = article_min10_2025_dataset.sort_values('article_mispricing_in_inr', ascending=False).head(5).copy()
    article_overvalued_2025_min10 = article_min10_2025_dataset.sort_values('article_mispricing_in_inr', ascending=True).head(5).copy()

pant_2025_review = top_5_overvalued_2025_realized_review.loc[top_5_overvalued_2025_realized_review['player_name'].eq('Rishabh Pant')]
if not pant_2025_review.empty:
    display(pant_2025_review)

top_5_undervalued_2025_realized_review, top_5_overvalued_2025_realized_review

## 14. Visual outputs

The charts below are exported to `outputs/charts/` using plain matplotlib.

In [ ]:
if not final_model_dataset.empty:
    plot_price_vs_predicted(article_2025_dataset if not article_2025_dataset.empty else final_model_dataset, OUTPUT_CHARTS / 'price_vs_predicted_price.png')
    plot_top_mispricing(article_2025_dataset if not article_2025_dataset.empty else final_model_dataset, OUTPUT_CHARTS / 'top_15_undervalued.png', kind='undervalued', top_n=5, mispricing_col='article_mispricing_in_inr', title='Top 5 Undervalued 2025 Auction Buys', subtitle='These players looked cheap relative to an article fair price built from cricket value plus an estimated market premium.')
    plot_top_mispricing(article_2025_dataset if not article_2025_dataset.empty else final_model_dataset, OUTPUT_CHARTS / 'top_15_overvalued.png', kind='overvalued', top_n=5, mispricing_col='article_mispricing_in_inr', title='Top 5 Overvalued 2025 Auction Buys', subtitle='These players looked expensive versus a fair price that includes both performance value and likely market premium.')
    plot_role_boxplot(final_model_dataset, OUTPUT_CHARTS / 'role_mispricing_boxplot.png')
    plot_franchise_efficiency(final_model_dataset, OUTPUT_CHARTS / 'franchise_value_efficiency.png')
    plot_quadrant(article_2025_dataset if not article_2025_dataset.empty else final_model_dataset, OUTPUT_CHARTS / 'quadrant_value_vs_price.png')

sorted(str(path.name) for path in OUTPUT_CHARTS.glob('*.png'))

## 15. Export outputs

The notebook writes the final modeling dataset and reporting tables to disk so the analysis can feed article writing, portfolio documentation, or downstream dashboards.

In [ ]:
export_frames = {
    'final_model_dataset.csv': final_model_dataset,
    'player_mispricing_table.csv': player_mispricing_table,
    'top_undervalued_players.csv': top_undervalued_players,
    'top_overvalued_players.csv': top_overvalued_players,
    'article_top_5_undervalued_2025.csv': article_top_5_undervalued_2025,
    'article_top_5_overvalued_2025.csv': article_top_5_overvalued_2025,
    'article_undervalued_2025_min10.csv': article_undervalued_2025_min10,
    'article_overvalued_2025_min10.csv': article_overvalued_2025_min10,
    'top_5_undervalued_2025_realized_review.csv': top_5_undervalued_2025_realized_review,
    'top_5_overvalued_2025_realized_review.csv': top_5_overvalued_2025_realized_review,
    'franchise_value_summary.csv': franchise_value_summary,
    'role_mispricing_summary.csv': role_mispricing_summary,
}

for file_name, frame in export_frames.items():
    if frame is None:
        frame = pd.DataFrame()
    frame.to_csv(OUTPUT_TABLES / file_name, index=False)

if not player_season_features.empty:
    player_season_features.to_csv(DATA_PROCESSED / 'player_season_features.csv', index=False)
if not name_review_df.empty:
    name_review_df.to_csv(DATA_PROCESSED / 'name_match_review.csv', index=False)
if not coverage_report.empty:
    coverage_report.to_csv(DATA_PROCESSED / 'coverage_report.csv', index=False)
if not comparison_df.empty:
    comparison_df.to_csv(DATA_PROCESSED / 'model_comparison.csv', index=False)

sorted(str(path.name) for path in OUTPUT_TABLES.glob('*.csv'))

## 16. Executive summary

This section condenses the notebook into portfolio-ready takeaways. If your raw data is not present yet, the text will stay sparse, but the structure remains reusable.

In [ ]:
summary_lines = []

if not comparison_df.empty:
    best_row = comparison_df.iloc[0]
    summary_lines.append(
        f"Primary fair-price model: {best_row['model']} with MAE(log price) {best_row['mae_log']:.3f}, RMSE(log price) {best_row['rmse_log']:.3f}, and R^2 {best_row['r2_log']:.3f}."
    )
if not top_undervalued_players.empty:
    row = top_undervalued_players.iloc[0]
    summary_lines.append(
        f"Most undervalued player in the sample: {row['player_name']} at {row['actual_price_in_crore']:.2f} crore versus a modeled fair price of {row['predicted_price_in_crore']:.2f} crore."
    )
if not top_overvalued_players.empty:
    row = top_overvalued_players.iloc[0]
    summary_lines.append(
        f"Most overvalued player in the sample: {row['player_name']} at {row['actual_price_in_crore']:.2f} crore versus a modeled fair price of {row['predicted_price_in_crore']:.2f} crore."
    )
if not franchise_value_summary.empty:
    row = franchise_value_summary.iloc[0]
    summary_lines.append(
        f"Best value-buying franchise by average captured mispricing: {row['team']} with {row['avg_positive_mispricing_captured'] / 10_000_000.0:.2f} crore of modeled surplus per player bought on average."
    )

summary_lines if summary_lines else ['Add raw data files to data/raw and rerun the notebook to populate the executive summary.']